# Initial Data Processing

This script shows how I do initial data processing before manipulating any GTFS datasets

There are FIVE parts in this notebook, producing NINE outputs (with their purposes in parentheses):
1) **Defining Study Areas (Origin) as Travel To Work Areas extended to 2021 MSOA Boundaries**
    - Getting a GPKG file of all MSOAs within study areas (for visualisation)
    - Getting a GPKG file with MSOA boundaries dissolved by study areas (for GTFS scoping)
    - Scoping OSM Protocol Buffer (PBF) Files to match the Study Area Boundaries (for r5py travel time modelling)

2) **Redefining City Centre (Destination) Boundaries by Extension to 2021 LSOA Boundaries**
    - Getting a GPKG file of city centre boundaries by study area (for GTFS scoping and visualisation)
    - Manipulating OD Flows (for calibration of Poisson GLM)

3) **Scoping Origin and Destination Points as 2021 LSOA Population Centroids within Study Areas/City Centre** (for r5py travel time modelling)

4) **Indices of Deprivation by MSOAs in Study Areas**
    - Getting a GPKG file of the most deprived MSOAs `imd_decile.isin([1, 2])` with boundaries dissolved by study areas (for GTFS scoping)

5) **Beefing up Study Area MSOAs with Additional Details** (for regression)
    - Median Euclidean Distance from Origin to Destination Points, aggregated to MSOA level (from Part 3)
    - Flows to city centre (from Part 3)
    - Population estimates
    - Bus lane infrastructure counts (from cropped OSM build in Part 1)
    - Road network characteristics
    - Car ownership per capita
---

The following datasets should be present in the 'Preprocessed' subfolder of the 'Data' directory before running this script:  

**FOR PART 1**
- [2011 Travel To Work Area - 2021 Output Area Best Fit Lookup Table](https://geoportal.statistics.gov.uk/datasets/ons::output-area-2021-to-ttwa-2011-to-lad-2022-best-fit-lookup-for-ew/about)  
- [2021 Output Area - LSOA - MSOA Lookup Table](https://geoportal.statistics.gov.uk/datasets/2713c3a627454ff88add814226264d77/about)
- [2021 MSOA Boundaries](https://geoportal.statistics.gov.uk/datasets/ons::middle-layer-super-output-areas-december-2021-boundaries-ew-bfc-v7-2/about)  
- [OpenStreetMap Data for England](https://download.geofabrik.de/europe/united-kingdom/england.html)


**FOR PART 2**
- CfC City Centre Boundaries (provided in GitHub)  
- [2011 Workplace Zone Population Centroids](https://geoportal.statistics.gov.uk/datasets/ons::workplace-zones-december-2011-ew-population-weighted-centroids-2/about)  
- [2011 Workplace Zone - 2011 Output Area Lookup Table](https://geoportal.statistics.gov.uk/datasets/ons::output-area-2011-to-workplace-zone-to-lad-december-2011-exact-fit-lookup-in-ew/about) 
- [2011 Output Area - 2021 Output Area Exact Fit Lookup Table](https://geoportal.statistics.gov.uk/datasets/ons::oa-2011-to-oa-2021-to-local-authority-district-2022-exact-fit-lookup-in-ew-v3/about)  
- [2021 LSOA Boundaries](https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bfc-v10-2/about)  
- [BRES Data for 2024 of all LSOAs and MSOAs](https://www.nomisweb.co.uk/datasets/newbres6pub)   
    - `Geography`: Select 'All' for LSOA and MSOA
    - `Date`: Select '2024'
    - `Employment Status`: Select 'Employees' only 
    - `Percent`: Select 'Count'
    - download as a CSV file  
- [Locomizer November 2021 Mobile Phone Data-Derived OD Flows](https://zenodo.org/records/13327082) (download `msoa_OD_allactivity.csv.gz`)


**FOR PART 3**
- [2021 LSOA Population Centroids](https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-ew-population-weighted-centroids-3/about)  


**FOR PART 4**
- [Indices of Deprivation 2025 at MSOA Level](https://github.com/JustKnowledge-UK/public-downloads/releases/download/data-release/iod25_msoa.csv)


**FOR PART 5**
- [Mid-2024 MSOA Population Estimates](https://www.ons.gov.uk/file?uri=/peoplepopulationandcommunity/populationandmigration/populationestimates/datasets/middlesuperoutputareamidyearpopulationestimates/mid2022revisednov2025tomid2024/sapemsoasyoa20222024.xlsx )
- [Ordnance Survey Road Dataset](https://osdatahub.os.uk/data/downloads/open/OpenRoads)
- [DfT 2025 Average Annual Daily Flow Data](https://storage.googleapis.com/dft-statistics/road-traffic/downloads/data-gov-uk/dft_traffic_counts_aadf.zip)

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import subprocess
import json
import re
from scipy.spatial import cKDTree

ROOT = Path("../Data")
ROOT.resolve()

# setting links for easy reference later
pre_processed = ROOT/'Preprocessed'
post_processed = ROOT/'Postprocessed'

# creating empty directories to store the outputs
os.makedirs(post_processed, exist_ok=True)

target_cities = ['Manchester', 'Bristol', 'Leeds']

In [ ]:
# loading all relevant datasets in the order listed above
# except for the OSM data, which is handled in a loop later

ttwa_oa21 = pd.read_csv(
    pre_processed/'Output_Area_(2021)_to_TTWAs_(2011)_to_LAD_(2022)_Lookup_for_England_and_Wales.csv',
    usecols=['TTWA11NM', 'OA21CD']
)

oa21_lsoa_msoa = pd.read_csv(
    pre_processed/'PCD_OA21_LSOA21_MSOA21_LAD_MAY26_UK_LU.csv',
    usecols=['oa21cd', 'lsoa21cd', 'msoa21cd']
)

msoa = gpd.read_file(
    pre_processed/'Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BFC_V7_1570414348754607696.gpkg'
).to_crs(27700)

city_centre = gpd.read_file(
    pre_processed/'2026-05-18_PUA_WZ-based_City_Centres.gpkg'
).to_crs(27700)

wz = gpd.read_file(
    pre_processed/'Workplace_Zones_Dec_2011_PWC_in_England_and_Wales_2022_6966230017104417570.gpkg'
).to_crs(27700)

wz_oa11 = pd.read_csv(
    pre_processed/'OA11_WZ11_LAD11_EW_LU_1e30e50424db4e97818d93e28fd2de3f_-869981087558068854.csv',
    usecols=['WZ11CD', 'OA11CD']
)

oa11_oa21 = pd.read_csv(
    pre_processed/'OA_(2011)_to_OA_(2021)_to_Local_Authority_District_(2022)_Exact_Fit_Lookup_in_EW_(V3).csv',
    usecols=['OA11CD', 'OA21CD']
)

lsoa = gpd.read_file(
    pre_processed/'Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BFC_V10_-672099234420024429.gpkg'
).to_crs(27700)

bres = pd.read_csv(
    pre_processed/'35889627457603.csv',
    usecols=['Area', 'Total']
)

od_flows = pd.read_csv(
    pre_processed/'msoa_ODs_allactivity.csv'
).rename(
    columns={
        'o_msoa': 'origin',
        'd_msoa': 'destination',
        'num_trips': 'count'
    }
)

lsoa_cent = gpd.read_file(
    pre_processed/'LSOA_PopCentroids_EW_2021_V4_-3471144733095659889.gpkg'
).to_crs(27700)

iod = pd.read_csv(
    pre_processed/'iod25_msoa.csv',
    usecols=[
        'msoa21cd', 
        'index_of_multiple_deprivation_imd_score_decile'
    ]
).rename(
    columns={
        'index_of_multiple_deprivation_imd_score_decile': 'imd_decile'
    }
)

popest = pd.read_csv(
    pre_processed/'mid2024_popest.csv',
    usecols=['MSOA 2021 Code', 'Total']
).rename(
    columns={
        'MSOA 2021 Code': 'MSOA21CD',
        'Total': 'popcount'
    }
)
popest['popcount'] = popest['popcount'].str.replace(',', '', regex=False).astype(int)

road_network = gpd.read_file(
    pre_processed/'oproad_gb.gpkg',
    layer='road_link',
    columns=['id', 'road_function', 'form_of_way', 'primary_route', 'geometry']
).to_crs(27700)

road_node = gpd.read_file(
    pre_processed/'oproad_gb.gpkg',
    layer='road_node',
    columns=['id', 'form_of_road_node', 'geometry']
).to_crs(27700)

stops = pd.read_csv(
    pre_processed/'Stops.csv',
    usecols=['ATCOCode', 'Longitude', 'Latitude', 'StopType', 'BusStopType', 'Status']
)
stops_gdf = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(
        stops['Longitude'],
        stops['Latitude']
    ),
    crs='EPSG:4326'
).to_crs(27700)

traffic_count = pd.read_csv(
    pre_processed/'dft_traffic_counts_aadf.csv',
    usecols=['count_point_id', 'year', 'latitude', 'longitude', 'all_motor_vehicles']
)
traffic_gdf = gpd.GeoDataFrame(
    traffic_count,
    geometry=gpd.points_from_xy(
        traffic_count['longitude'],
        traffic_count['latitude']
    ),
    crs='EPSG:4326'
).to_crs(27700)

In [ ]:
# creating three major lookup tables

# Lookup 1: WZ --> OA 2011 --> OA 2021 --> LSOA 2021
wz_to_lsoa = pd.merge(
    wz_oa11,
    oa11_oa21,
    on='OA11CD',
    how='inner'
).merge(
    oa21_lsoa_msoa,
    left_on='OA21CD',
    right_on='oa21cd',
    how='inner'
)

# Lookup 2: TTWA --> OA 2021 --> MSOA 2021
ttwa_to_msoa = pd.merge(
    ttwa_oa21,
    oa21_lsoa_msoa,
    left_on='OA21CD',
    right_on='oa21cd',
    how='inner'
)

oa11_to_msoa21 = pd.merge(
    oa11_oa21,
    oa21_lsoa_msoa,
    left_on='OA21CD',
    right_on='oa21cd',
    how='inner'
)

## Part 1: Defining Study Areas

Study Areas will be MSOAs that intersect with TTWAs for Manchester, Leeds and Bristol.

### 1.1: GPKG File of Study Area MSOAs with boundaries dissolved by Study Areas

In [ ]:
# Output 1: Study Area as a Collection of MSOAs
filtered_ttwa_to_msoa = ttwa_to_msoa[ttwa_to_msoa['TTWA11NM'].isin(target_cities)]

updated_msoa = msoa.merge(
    filtered_ttwa_to_msoa.drop_duplicates(subset=['msoa21cd']),
    left_on='MSOA21CD',
    right_on='msoa21cd',
    how='left'
).dropna(
    subset=['TTWA11NM']
)
updated_msoa = updated_msoa[['MSOA21CD', 'TTWA11NM', 'geometry']]
updated_msoa['area_km2'] = updated_msoa.area / 1000000

updated_studyarea = []

for city in target_cities:
    updated_studyarea.append({
        'study_area': city,
        'geometry': updated_msoa[updated_msoa['TTWA11NM'] == city].union_all()
    })

updated_studyarea = gpd.GeoDataFrame(
    updated_studyarea,
    crs='EPSG:27700'
)

updated_studyarea.to_file(post_processed/'updated_studyarea.gpkg', driver='GPKG')
print("Updated Study Area file created successfully.")

### 1.2: Scoping Down OSM PBF Files

r5py requires minimally a GTFS dataset and a OpenStreetMap network in PBF file format. Thus, the England-wide OSM PBF file needs to be scoped down such that it only covers the study areas as defined earlier.

This requires the use of `osmium`, a C++ library. Python's [`subprocess`](geeksforgeeks.org/python/python-subprocess-module/) is the bridge that allows us to run non-Python operations from Python.

In [ ]:
for_osmclipping = updated_studyarea.copy().to_crs(4326)

# Write each city's polygon as a GeoJSON for osmium to use
for _, row in for_osmclipping.iterrows():
    city = row['study_area']
    
    # osmium extract needs a GeoJSON file with the clip polygon
    clip_geojson = {
        "type": "Feature",
        "geometry": row['geometry'].__geo_interface__
    }
    
    clip_path = post_processed / f'{city}_clip.geojson'
    with open(clip_path, 'w') as f:
        json.dump(clip_geojson, f)
    
    output_pbf = post_processed/f'{city}_ttwa.osm.pbf'
    
    result = subprocess.run(
        [
            'osmium', 'extract',
            '--polygon', str(clip_path),
            '--strategy', 'complete_ways',
            str(pre_processed/'england-260627.osm.pbf'),
            '-o', str(output_pbf),
            '--overwrite'
        ], 
        capture_output=True, 
        text=True
    )
    
    if result.returncode != 0:
        print(f"ERROR for {city}:\n{result.stderr}")
    else:
        print(f"OSM extract for {city} done")

## Part 2: Redefining City Centre Boundaries

City Centres will be redefined as the LSOAs that intersect with the original city centre boundaries defined in CfC's report.

### 2.1: GPKG File of City Centre Boundaries by Study Area

In [ ]:
lsoa_list = []
updated_city_centre = []

for city in target_cities:
    cbd = city_centre[city_centre['pua'] == city]
    
    cbd_wz = gpd.sjoin(
        wz,
        cbd,
        how='left',
        predicate='within'
    ).dropna(
        subset=['pua']
    )
    
    filtered_wz_to_lsoa = wz_to_lsoa[wz_to_lsoa['WZ11CD'].isin(cbd_wz['wz11cd'])]

    lsoa_list.append(filtered_wz_to_lsoa['lsoa21cd'].unique())
    
    updated_city_centre.append({
        'pua': city,
        'geometry': lsoa[lsoa['LSOA21CD'].isin(filtered_wz_to_lsoa['lsoa21cd'].unique())].union_all()
    })

full_lsoa_list = np.concatenate(lsoa_list)

updated_city_centre = gpd.GeoDataFrame(
    updated_city_centre,
    crs='EPSG:27700'
)

updated_city_centre.to_file(post_processed/'updated_city_centre.gpkg', driver='GPKG')
print("Updated City Centre file created successfully.")

### 2.2: Processing OD Flows Data

Since OD flow data is about MSOA-to-MSOA flows, but the city centre is defined by LSOA boundaries, we need to adjust the flows to city centre. Here's how I did it:

1) Find out the ratio of jobs in city centre for city centre-intersected MSOAs using BRES data

2) Multiply the flow of origin MSOAs to city centre-intersected MSOAs by that ratio

3) Aggregate the flows by origin MSOAs to generate a combined flow to city centre from each origin MSOA

In [ ]:
# STEP 1: GENERATING THE RATIO OF CITY CENTRE JOBS

# Cleaning the BRES dataset
bres[['type', 'code', 'name']] = bres['Area'].str.split(':', expand=True).apply(lambda col: col.str.strip())
bres = bres[['type', 'code', 'Total']]

# Splitting it for BRES data of LSOAs and MSOAs
lsoa_bres = bres[bres['type'] == 'lsoa2021']
msoa_bres = bres[bres['type'] == 'msoa2021']

# Filtering for LSOA BRES data within Redefined City Centre Boundaries
citycentre_emp = lsoa_bres[lsoa_bres['code'].isin(full_lsoa_list)]

# Merging it with MSOA BRES data
emp_combined = citycentre_emp.merge(
    oa21_lsoa_msoa.drop_duplicates(subset=['lsoa21cd']),
    left_on='code',
    right_on='lsoa21cd',
    how='left'
).merge(
    msoa_bres,
    left_on='msoa21cd',
    right_on='code',
    how='left'
).rename(
    columns={
        'Total_x': 'LSOA_count',
        'Total_y': 'MSOA_count'
    }
).drop(
    columns=['code_x', 'code_y', 'type_x', 'type_y', 'oa21cd']
)

# Getting the ratio of jobs within city centre
emp_ratio = emp_combined.groupby('msoa21cd').agg(
    in_cc=('LSOA_count', 'sum'),
    total=('MSOA_count', 'first')
).reset_index().merge(
    updated_msoa,
    left_on='msoa21cd',
    right_on='MSOA21CD',
    how='left'
)

emp_ratio['ratio'] = emp_ratio['in_cc'] / emp_ratio['total']
emp_ratio = emp_ratio[['MSOA21CD', 'TTWA11NM', 'in_cc', 'total', 'ratio']]

In [ ]:
updated_odflows = []

# STEPS 2 AND 3: MULTIPLYING THE OD FLOWS BY THE RATIO OF CITY CENTRE JOBS AND AGGREGATING THEM BY ORIGIN MSOA
for city in target_cities:

    # scoping postprocessed datasets to target city
    scoped_studyarea = updated_msoa[updated_msoa['TTWA11NM'] == city].copy()
    scoped_ratio = emp_ratio[emp_ratio['TTWA11NM'] == city]

    # scoping OD flows
    origscoped_odflows = od_flows[od_flows['origin'].isin(scoped_studyarea['MSOA21CD'].unique())].copy()
    bothscoped_odflows = origscoped_odflows[origscoped_odflows['destination'].isin(scoped_ratio['MSOA21CD'].unique())].copy()

    # establishing a lookup 
    jobratio_lookup = scoped_ratio.set_index('MSOA21CD')['ratio'].to_dict()

    # get the estimated flows 
    def est_count(row):
        target_msoa = row['destination']

        if target_msoa in jobratio_lookup:
            return round(row['count'] * jobratio_lookup[target_msoa])

        return 0

    bothscoped_odflows['est_count'] = bothscoped_odflows.apply(est_count, axis=1)

    # aggregating the estimated flows by origin MSOA
    processed_odflows = bothscoped_odflows.groupby(
        'origin'
    ).agg(
        count = ('est_count', 'sum'),
    ).reset_index()

    # append the list of estimated flows
    updated_odflows.append(
        processed_odflows
    )

updated_odflows = pd.concat(
    updated_odflows
).rename(
    columns={'origin': 'MSOA21CD'}
)
# This will be merged with Study Area MSOA details later in Part 5

## Part 3: Defining Origin and Destination Points for Transport Modelling

The points will be based on LSOA population centroids. We will have two outputs - one for all origin points and another for all destination points

In [ ]:
# Output 1: All Origin Points
origin_points = lsoa_cent.merge(
    filtered_ttwa_to_msoa.drop_duplicates(subset=['lsoa21cd']),
    left_on='LSOA21CD',
    right_on='lsoa21cd',
    how='left'
).dropna(
    subset=['TTWA11NM']
)
origin_points = origin_points[['LSOA21CD', 'msoa21cd', 'TTWA11NM', 'geometry']]

origin_points.to_file(post_processed/'origin_points.gpkg', driver='GPKG')
print("Origin Points file created successfully.")

In [ ]:
# Output 2: All Destination Points
citycentre_ttwa_to_msoa = filtered_ttwa_to_msoa[filtered_ttwa_to_msoa['lsoa21cd'].isin(full_lsoa_list)]

destination_points = lsoa_cent.merge(
    citycentre_ttwa_to_msoa.drop_duplicates(subset=['lsoa21cd']),
    left_on='LSOA21CD',
    right_on='lsoa21cd',
    how='left'
).dropna(
    subset=['TTWA11NM']
)
destination_points = destination_points[['LSOA21CD', 'msoa21cd', 'TTWA11NM', 'geometry']]

destination_points.to_file(post_processed/'destination_points.gpkg', driver='GPKG')
print("Destination Points file created successfully.")

In [ ]:
# getting euclidean distance from origin to destination points
# to be used later in Part 5

all_distances = []

for city in target_cities:

    origin_city = origin_points[origin_points['TTWA11NM'] == city]
    dest_city = destination_points[destination_points['TTWA11NM'] == city]

    # Cartesian product
    pairs = (
        origin_city.assign(key=1)
        .merge(dest_city.assign(key=1), on='key', suffixes=('_orig', '_dest'))
        .drop(columns='key')
    )

    # Euclidean distance
    pairs['distance'] = (
        pairs.geometry_orig.distance(pairs.geometry_dest)
    )

    all_distances.append(pairs)

distance_matrix = pd.concat(
    all_distances, 
    ignore_index=True
).groupby(
    'msoa21cd_orig'
).agg(
    median_eucldist = ('distance', 'median')
).reset_index().rename(
    columns={'msoa21cd_orig': 'MSOA21CD'}
)

distance_matrix['median_eucldist_km'] = distance_matrix['median_eucldist'] / 1000

## Part 4: Indices of Deprivation 2025 at MSOA Level

### 4.1: GPKG of Most Deprived MSOAs by Study Area

In [ ]:
msoa_iod = updated_msoa.merge(
    iod,
    left_on='MSOA21CD',
    right_on='msoa21cd',
    how='left'
).drop(
    columns=['msoa21cd']
)

group_labels = {
    1: 'Most deprived', 
    2: 'Most deprived', 
    3: '20-40 pctile', 
    4: '20-40 pctile', 
    5: '40-60 pctile',
    6: '40-60 pctile',
    7: '60-80 pctile',
    8: '60-80 pctile',
    9: 'Least deprived',
    10: 'Least deprived'
}

msoa_iod['imd_group_label'] = msoa_iod['imd_decile'].map(group_labels)

filtered_iod = msoa_iod[msoa_iod['imd_group_label'] == 'Most deprived'].copy()

deprived_msoa = []

for city in target_cities:
    deprived_msoa.append({
        'study_area': city,
        'geometry': filtered_iod[filtered_iod['TTWA11NM'] == city].union_all()
    })

deprived_msoa = gpd.GeoDataFrame(
    deprived_msoa,
    crs='EPSG:27700'
)

deprived_msoa.to_file(post_processed/'deprived_msoa.gpkg', driver='GPKG')
print("Deprived MSOAs GPKG file created successfully.")

## Part 5: Beefing up Study Area MSOAs with Additional Details

### 5.1: Add Median Euclidean Distance to City Centre

In [ ]:
msoa_wdist = msoa_iod.merge(
    distance_matrix[['MSOA21CD', 'median_eucldist_km']],
    on='MSOA21CD',
    how='left'
)

### 5.2: Add City Centre Flows

In [ ]:
msoa_wflows = msoa_wdist.merge(
    updated_odflows,
    on='MSOA21CD',
    how='left'
)

### 5.3: Add Mid-2024 Population Estimates

In [ ]:
msoa_wpopest = msoa_wflows.merge(
    popest,
    on='MSOA21CD',
    how='left'
)

### 5.4: Prepare Bus Lane Infrastructure per MSOA

In [ ]:
# STEP 1: Creating a GDF of all bus lanes in the target cities

def extract_tag(other_tags, key):
    if pd.isna(other_tags):
        return None
    match = re.search(rf'"{re.escape(key)}"=>"([^"]*)"', other_tags)
    return match.group(1) if match else None

full_buslanes = []

for city in target_cities:

    osm_roads = gpd.read_file(
        post_processed/f'{city}_ttwa.osm.pbf', 
        layer='lines',
        columns=['osm_id', 'highway', 'other_tags', 'geometry']
    ).to_crs(4326)

    for tag in ["lanes:bus", "lanes:psv", "bus:lanes", "psv:lanes", "cycleway"]:
        osm_roads[tag] = osm_roads["other_tags"].apply(lambda x: extract_tag(x, tag))

    bus_lane_mask = (
        osm_roads["highway"].isin(["busway", "bus_guideway"])
        | osm_roads["lanes:bus"].notna()
        | osm_roads["lanes:psv"].notna()
        | osm_roads["bus:lanes"].notna()
        | osm_roads["psv:lanes"].notna()
        | (osm_roads["cycleway"] == "share_busway")
    )

    bus_lanes = osm_roads[bus_lane_mask]

    full_buslanes.append(bus_lanes)

full_buslanes = gpd.GeoDataFrame(
    pd.concat(full_buslanes, ignore_index=True)[['osm_id', 'geometry']],
    crs='EPSG:4326'
).to_crs(27700)

In [ ]:
# STEP 2: Finding out how many bus lanes in study area MSOAs based on intersection

buslane_msoa_intersect = gpd.sjoin(
    full_buslanes,
    msoa_wpopest[['MSOA21CD', 'geometry']]
).groupby(
    'MSOA21CD'
).agg(
    buslane_count=('osm_id', 'count')
).reset_index()

### 5.5: Add Road Network Characteristics per MSOA

In [ ]:
# STEP 1: One-hot encoding the road network data

# One-hot encode the two categorical columns
road_onehot = pd.get_dummies(
    road_network,
    columns=['road_function', 'form_of_way'],
    dtype=int
)

road_onehot['Dual'] = road_onehot['form_of_way_Collapsed Dual Carriageway'] + road_onehot['form_of_way_Dual Carriageway']

road_onehot['primary_route'] = road_onehot['primary_route'].astype(int)

In [ ]:
# STEP 2: Filter for roads within study area MSOAs

road_msoa_intersect = gpd.sjoin(
    road_onehot,
    msoa_wpopest[['MSOA21CD', 'geometry']]
)

In [ ]:
# STEP 3: Creating summary statistics about road network characteristics for each MSOA
# Add the bus lane info from Part 5.4

road_stats = road_msoa_intersect.groupby(
    'MSOA21CD'
).agg(
    count = ('id', 'count'),
    minorrd_count = ('road_function_Minor Road', 'sum'),
    broad_count = ('road_function_B Road', 'sum'),
    aroad_count = ('road_function_A Road', 'sum'),
    dual_count = ('Dual', 'sum'),
    primary_count = ('primary_route', 'sum')
).reset_index().merge(
    buslane_msoa_intersect,
    on='MSOA21CD',
    how='left'
)

road_stats['minor_rd'] = road_stats['minorrd_count'] / road_stats['count']
road_stats['b_road'] = road_stats['broad_count'] / road_stats['count']
road_stats['a_road'] = road_stats['aroad_count'] / road_stats['count']
road_stats['dual'] = road_stats['dual_count'] / road_stats['count']
road_stats['primary'] = road_stats['primary_count'] / road_stats['count']
road_stats['bus_lane'] = (road_stats['buslane_count'] / road_stats['count']).fillna(0)

In [ ]:
# STEP 4: Merge that to the MSOA dataset

msoa_wroadstats = msoa_wpopest.merge(
    road_stats[['MSOA21CD', 'minor_rd', 'b_road', 'a_road', 'dual', 'primary', 'bus_lane']],
    on='MSOA21CD',
    how='left'
)

### 5.6: Add Junction, Bus Stop and Rail/Tram Stop Density per MSOA

In [ ]:
# STEP 1: Filter for junctions in study area and merge

junction = road_node[road_node['form_of_road_node'].isin(['junction', 'roundabout'])]

junction_msoa_intersect = gpd.sjoin(
    junction,
    msoa_wroadstats[['MSOA21CD', 'geometry']]
).groupby(
    'MSOA21CD'
).agg(
    junction_count=('id', 'count')
).reset_index()

In [ ]:
# STEP 2: Filter for bus stops in study area

active_stops = stops_gdf[stops_gdf['Status'] == 'active']
bus_stops = active_stops[active_stops['StopType'] == 'BCT']
marked_stops = bus_stops[bus_stops['BusStopType'].isin(['MKD', 'CUS'])]

busstop_msoa_intersect = gpd.sjoin(
    marked_stops,
    msoa_wroadstats[['MSOA21CD', 'geometry']]
).groupby(
    'MSOA21CD'
).agg(
    busstop_count=('ATCOCode', 'count')
).reset_index()

In [ ]:
# STEP 3: Merge with the MSOA dataset and find density of junctions/stops per km^2

msoa_wpoints = msoa_wroadstats.merge(
    junction_msoa_intersect,
    on='MSOA21CD',
    how='left'
).merge(
    busstop_msoa_intersect,
    on='MSOA21CD',
    how='left'
)

msoa_wpoints['junction_dens'] = msoa_wpoints['junction_count'] / msoa_wpoints['area_km2']
msoa_wpoints['busstop_dens'] = msoa_wpoints['busstop_count'] / msoa_wpoints['area_km2']

### 5.7: Add Per KM Traffic Count Per MSOA

In [ ]:
# STEP 1: Filter for 2025 AADF counts within study area, then find vehicle count per km of road

aadf_2025 = traffic_gdf[traffic_gdf['year'] == 2025]

mask = updated_studyarea.copy()
mask.geometry = mask.buffer(2000)

aadf_cropped = gpd.sjoin(
    aadf_2025,
    mask,
    how='inner',
    predicate='within'
).dropna(
    subset=['study_area']
)

nodes_inmsoa = gpd.sjoin(
    road_node,
    msoa_wpoints[['MSOA21CD', 'geometry']]
).dropna(
    subset=['MSOA21CD']
)

In [ ]:
# STEP 2: Interpolate the AADF counts to all nodes within the MSOA

def idw_interpolate(source_coords, source_values, target_coords, k=10, power=2, epsilon=1e-6):
    tree = cKDTree(source_coords)
    dist, idx = tree.query(target_coords, k=k)

    # nudge zero-distance (exact overlap) cases so 1/dist doesn't divide by zero
    dist = np.where(dist == 0, epsilon, dist)

    weights = 1 / dist**power
    weights /= weights.sum(axis=1, keepdims=True)  # normalize so weights sum to 1

    return np.sum(weights * source_values[idx], axis=1)

source_coords = np.column_stack([aadf_cropped.geometry.x, aadf_cropped.geometry.y])
source_values = aadf_cropped['all_motor_vehicles'].values

target_coords = np.column_stack([nodes_inmsoa.geometry.x, nodes_inmsoa.geometry.y])

nodes_inmsoa['aadf_interp'] = idw_interpolate(source_coords, source_values, target_coords, k=10)

In [ ]:
# STEP 3: Merging that to the MSOA dataset

msoa_traffic = nodes_inmsoa.groupby(
    'MSOA21CD'
).agg(
    median_aadf = ('aadf_interp', 'median')
).reset_index()

fullmsoa_details = msoa_wpoints.merge(
    msoa_traffic,
    on='MSOA21CD',
    how='left'
)

fullmsoa_details['railstop_dens'] = fullmsoa_details['railstop_dens'].fillna(0)

fullmsoa_details.to_file(post_processed/'updated_msoa.gpkg', driver='GPKG')
fullmsoa_details